In [0]:
%pip install --upgrade langchain_community pypdf typing_extensions

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

volume_path = '/Volumes/llm/rag/healthcare_docs'
pdf_files = [os.path.join(volume_path, f) for f in os.listdir(volume_path) if f.lower().endswith('.pdf')]

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100)

    docs = splitter.split_documents(documents)

    doc_chunk_list = []

    for i, d in enumerate(docs):
        doc_chunk_list.append(
            {
                'chunk_id': i + 1,
                'metadata': str(d.metadata),
                'text': d.page_content
            }
        )

    df = spark.createDataFrame(doc_chunk_list)
    from pyspark.sql.functions import lit

    file_id = pdf_file
    df = df.withColumn("file_id", lit(file_id))

    from delta.tables import DeltaTable

    table_name = 'llm.rag.docs_text'

    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        delta_table.alias("t").merge(
            df.alias("s"),
            "t.chunk_id = s.chunk_id and t.file_id = s.file_id"
        ).whenNotMatchedInsert(values={
            'chunk_id': 's.chunk_id',
            'metadata': 's.metadata',
            'text': 's.text',
            'file_id': 'file_id'
        }).execute()
    else:
        df.select('chunk_id', 'metadata', 'text', 'file_id').write.format('delta').mode('overwrite').saveAsTable(table_name)